# Heart Disease Prediction — ML Mini Project
**Dataset:** Cleveland Heart Disease  
**Models:** Logistic Regression & Random Forest

## Setup & Imports
Import all required libraries.

In [ ]:
# =============================================================
#  Heart Disease Prediction - ML Mini Project
#  Dataset: Cleveland Heart Disease
#  Models: Logistic Regression & Random Forest
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

## 1. Load Data
Load the Cleveland Heart Disease dataset and inspect its structure.

In [ ]:
# ── 1. LOAD DATA ──────────────────────────────────────────────
df = pd.read_csv("heart_cleveland_upload.csv")

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Shape          : {df.shape}")
print(f"Target classes : {df['condition'].value_counts().to_dict()}  (0=No Disease, 1=Disease)")
print("\nFirst 5 rows:")
print(df.head())
print("\nMissing values:", df.isnull().sum().sum())

## 2. Exploratory Data Analysis
Visualise class distribution and feature correlations.

In [ ]:
# ── 2. EXPLORATORY DATA ANALYSIS ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
df['condition'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','tomato'],
                                     edgecolor='black')
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['No Disease (0)', 'Disease (1)'], rotation=0)
axes[0].set_ylabel('Count')

# Correlation heatmap
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".1f", cmap='coolwarm', ax=axes[1], linewidths=0.5)
axes[1].set_title('Feature Correlation Heatmap')

plt.tight_layout()
plt.savefig("eda_plots.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n[EDA plot saved as eda_plots.png]")

## 3. Feature / Target Split & Train-Test Split
Split features and target, then create train/test sets with stratification. Apply `StandardScaler` for Logistic Regression.

In [ ]:
# ── 3. FEATURE / TARGET SPLIT ─────────────────────────────────
X = df.drop('condition', axis=1)
y = df['condition']

feature_names = X.columns.tolist()
print(f"\nFeatures ({len(feature_names)}): {feature_names}")

# ── 4. TRAIN / TEST SPLIT ─────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")

# ── 5. FEATURE SCALING (for Logistic Regression) ──────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## Model 1 — Logistic Regression
Train Logistic Regression, evaluate with accuracy, ROC-AUC, classification report, and 5-fold cross-validation.

In [ ]:
# =============================================================
#  MODEL 1 — LOGISTIC REGRESSION
# =============================================================
print("\n" + "=" * 60)
print("MODEL 1: LOGISTIC REGRESSION")
print("=" * 60)

lr_model = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
lr_model.fit(X_train_scaled, y_train)

y_pred_lr    = lr_model.predict(X_test_scaled)
y_prob_lr    = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_acc  = accuracy_score(y_test, y_pred_lr)
lr_auc  = roc_auc_score(y_test, y_prob_lr)

print(f"\nAccuracy  : {lr_acc:.4f}  ({lr_acc*100:.2f}%)")
print(f"ROC-AUC   : {lr_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['No Disease', 'Disease']))

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print(f"5-Fold CV Accuracy: {lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}")

# Feature coefficients
lr_coef_df = pd.DataFrame({
    'Feature'    : feature_names,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)
print("\nTop Feature Coefficients:")
print(lr_coef_df.to_string(index=False))

## Model 2 — Random Forest
Train Random Forest (no scaling needed), evaluate similarly.

In [ ]:
# =============================================================
#  MODEL 2 — RANDOM FOREST
# =============================================================
print("\n" + "=" * 60)
print("MODEL 2: RANDOM FOREST")
print("=" * 60)

rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=None,
    min_samples_split=2, random_state=42
)
rf_model.fit(X_train, y_train)   # RF doesn't need scaling

y_pred_rf = rf_model.predict(X_test)
y_prob_rf  = rf_model.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_prob_rf)

print(f"\nAccuracy  : {rf_acc:.4f}  ({rf_acc*100:.2f}%)")
print(f"ROC-AUC   : {rf_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['No Disease', 'Disease']))

# Cross-validation
rf_cv_scores = cross_val_score(rf_model, X_train, y_train, cv=cv, scoring='accuracy')
print(f"5-Fold CV Accuracy: {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}")

# Feature importances
rf_feat_df = pd.DataFrame({
    'Feature'   : feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print("\nFeature Importances:")
print(rf_feat_df.to_string(index=False))

## Evaluation Plots
Confusion matrices, ROC curves, feature importance/coefficients, and accuracy comparison.

In [ ]:
# =============================================================
#  PLOTS — CONFUSION MATRICES, ROC CURVES, FEATURE IMPORTANCE
# =============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Heart Disease Prediction — Model Evaluation', fontsize=14, fontweight='bold')

# ── Confusion Matrix: LR ──
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lr),
                       display_labels=['No Disease', 'Disease']).plot(
    ax=axes[0][0], colorbar=False, cmap='Blues'
)
axes[0][0].set_title('Logistic Regression\nConfusion Matrix')

# ── Confusion Matrix: RF ──
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_rf),
                       display_labels=['No Disease', 'Disease']).plot(
    ax=axes[0][1], colorbar=False, cmap='Greens'
)
axes[0][1].set_title('Random Forest\nConfusion Matrix')

# ── ROC Curve Comparison ──
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
axes[0][2].plot(fpr_lr, tpr_lr, label=f'Logistic Reg (AUC={lr_auc:.2f})', color='steelblue', lw=2)
axes[0][2].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={rf_auc:.2f})', color='tomato',    lw=2)
axes[0][2].plot([0,1],[0,1], 'k--', lw=1)
axes[0][2].set_xlabel('False Positive Rate')
axes[0][2].set_ylabel('True Positive Rate')
axes[0][2].set_title('ROC Curve Comparison')
axes[0][2].legend(loc='lower right')

# ── LR Feature Coefficients ──
colors_lr = ['tomato' if c > 0 else 'steelblue' for c in lr_coef_df['Coefficient']]
axes[1][0].barh(lr_coef_df['Feature'], lr_coef_df['Coefficient'], color=colors_lr, edgecolor='black')
axes[1][0].axvline(0, color='black', linewidth=0.8)
axes[1][0].set_title('Logistic Regression\nFeature Coefficients')
axes[1][0].invert_yaxis()

# ── RF Feature Importances ──
axes[1][1].barh(rf_feat_df['Feature'], rf_feat_df['Importance'], color='mediumseagreen', edgecolor='black')
axes[1][1].set_title('Random Forest\nFeature Importances')
axes[1][1].invert_yaxis()

# ── Accuracy Comparison Bar ──
models     = ['Logistic Regression', 'Random Forest']
accuracies = [lr_acc, rf_acc]
aucs       = [lr_auc, rf_auc]
x = np.arange(len(models))
w = 0.35
bars1 = axes[1][2].bar(x - w/2, accuracies, w, label='Accuracy', color='cornflowerblue', edgecolor='black')
bars2 = axes[1][2].bar(x + w/2, aucs,       w, label='ROC-AUC',  color='salmon',         edgecolor='black')
for bar in bars1 + bars2:
    axes[1][2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
axes[1][2].set_xticks(x)
axes[1][2].set_xticklabels(models)
axes[1][2].set_ylim(0, 1.1)
axes[1][2].set_title('Model Comparison\n(Accuracy vs ROC-AUC)')
axes[1][2].legend()

plt.tight_layout()
plt.savefig("model_evaluation.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n[Evaluation plot saved as model_evaluation.png]")

## Final Summary
Print a consolidated summary table and declare the best model by ROC-AUC.

In [ ]:
# =============================================================
#  FINAL SUMMARY
# =============================================================
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"{'Model':<25} {'Accuracy':>10} {'ROC-AUC':>10} {'CV Acc (mean)':>15}")
print("-" * 62)
print(f"{'Logistic Regression':<25} {lr_acc:>10.4f} {lr_auc:>10.4f} {lr_cv_scores.mean():>15.4f}")
print(f"{'Random Forest':<25} {rf_acc:>10.4f} {rf_auc:>10.4f} {rf_cv_scores.mean():>15.4f}")
print("=" * 60)
winner = "Random Forest" if rf_auc >= lr_auc else "Logistic Regression"
print(f"\n✔  Best model by ROC-AUC: {winner}")